Cleanse 1m price data and write data to silver layer table 1m_prices_cleansed

In [0]:
from pyspark.sql.functions import col, when, explode, map_values
from datetime import date
from dateutil.relativedelta import relativedelta
from delta.tables import *
from time import time

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Get data from '{catalog}.01_bronze.1m_prices_raw' remove duplicates
df_1m_prices = spark.read.table(f"{catalog}.01_bronze.1m_prices_raw").dropDuplicates()

# filter data to only last 15 mintutes and remove prices for items we will never be trading
# job will run this notebook every 10 minutes
# TODO re-evaluate the price filter
unix_timestamp = int(time())
df_1m_prices = df_1m_prices.filter(f"(time > {unix_timestamp} - 900)")

In [0]:
# Insert df_1m_prices into {catalog}.02_silver.1m_prices_cleansed when id, time, and highorlow are not matched

targetDF = DeltaTable.forName(spark, f"{catalog}.02_silver.1m_prices_cleansed")
dfUpdates = df_1m_prices

targetDF.alias("t") .\
  merge(
    source = dfUpdates.alias("s"),
    condition = "t.id = s.id AND t.time = s.time AND \
         t.highorlow = s.highorlow") .\
  whenNotMatchedInsertAll() .\
  execute()
